# Advanced TwoTower training with item features


The standard [TwoTower example](15_twotower_example.ipynb) shows the simplest setup with `item_id` and `TwoTower.from_params`. This advanced example adds the parts needed for a more flexible and efficient training setup:

- `genre` and `movie_title` item features, including their preprocessing and encoding;
- an explicit `ParquetModule` transform pipeline instead of a default transform factory;
- independent negative pools for logical groups packed into one larger physical batch;
- target-aware trimming and length-bucketed query encoding to reduce work on left padding;
- a `TwoTower` model assembled from individual building blocks with a custom encoder and loss;
- optional chunked item-cache construction to reduce peak GPU memory use.

The example uses `ParquetModule` to load and process data.

Set `ARROW_NUM_THREADS` and `ARROW_IO_THREADS` to the number of available CPU cores. Set the OpenMP, MKL, and PyTorch thread counts to `1` so that they do not compete with Arrow workers.


In [1]:
%env OMP_NUM_THREADS=1
%env MKL_NUM_THREADS=1
%env ARROW_NUM_THREADS=8
%env ARROW_IO_THREADS=8

env: OMP_NUM_THREADS=1
env: MKL_NUM_THREADS=1
env: ARROW_NUM_THREADS=8
env: ARROW_IO_THREADS=8


In [2]:
import lightning as L  # noqa: N812
import pandas as pd
import torch

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

L.seed_everything(42)

Seed set to 42


42

## Data preparation

### Split and encode the data


You can prepare the data with Pandas, Polars, PySpark, or another tool. `ParquetModule` only requires the final data to be stored in Parquet files. This example uses Pandas.

The example uses MovieLens-1M from `examples/data`. The model uses `item_id` from the interactions and two movie features: `movie_title` and `genre`.


In [3]:
user_features = pd.read_csv(
    "data/ml1m_users.dat",
    sep="\t",
    names=["user_id", "gender", "age", "occupation", "zip_code"],
)[["user_id"]]
item_features = pd.read_csv(
    "data/ml1m_items.dat",
    sep="\t",
    names=["item_id", "movie_title", "genre"],
)
interactions = pd.read_csv(
    "data/ml1m_ratings.dat",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
).drop(columns=["rating"])


In [4]:
item_features.head(5)

,item_id,movie_title,genre
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
interactions["timestamp"] = interactions["timestamp"].astype("int64")
interactions = interactions.sort_values(by="timestamp")
interactions["timestamp"] = interactions.groupby("user_id").cumcount()
interactions.head(5)

,user_id,item_id,timestamp
1000138,6040,858,0
999873,6040,593,1
1000153,6040,2384,2
1000192,6040,2019,3
1000007,6040,1961,4


Split the interactions into train, validation, and test sets with the last-one-out strategy. For simplicity, remove cold users and items from validation and test data.


In [6]:
from replay.splitters import LastNSplitter

splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True,
)

raw_test_events, raw_test_gt = splitter.split(interactions)
raw_val_events, raw_val_gt = splitter.split(raw_test_events)
raw_train_events = raw_val_events

Fit a `LabelEncoder` for `user_id` and `item_id`. We fit it on the user and item catalogs so that validation and test data do not contain unknown IDs.

Another option is to fit the encoder on the training data and call `partial_fit` before testing when new IDs must be supported.

Use the fitted encoder to transform the interaction data.


In [7]:
from replay.preprocessing import LabelEncoder, LabelEncodingRule

encoder = LabelEncoder(
    [
        LabelEncodingRule("user_id"),
        LabelEncodingRule("item_id"),
    ]
)

encoder.rules[0].fit(user_features)
encoder.rules[1].fit(item_features)

train_events = encoder.transform(raw_train_events)
val_events = encoder.transform(raw_val_events)
val_gt = encoder.transform(raw_val_gt)
test_events = encoder.transform(raw_test_events)
test_gt = encoder.transform(raw_test_gt)

### Prepare item features

Encode `item_id` in the movie catalog with the fitted `LabelEncoder`.

The model will also use `movie_title` and `genre`, so these features need to be encoded as well.


In [8]:
item_features_encoded = encoder.rules[1].transform(item_features)
item_features_encoded.head(5)

,movie_title,genre,item_id
0,Toy Story (1995),Animation|Children's|Comedy,0
1,Jumanji (1995),Adventure|Children's|Fantasy,1
2,Grumpier Old Men (1995),Comedy|Romance,2
3,Waiting to Exhale (1995),Comedy|Drama,3
4,Father of the Bride Part II (1995),Comedy,4


RePlay transformer models support four feature types. Set the type with `feature_type` in `TensorSchema`:

- `CATEGORICAL`: one categorical value per object, such as an item ID.
- `CATEGORICAL_LIST`: several categorical values per object, such as movie genres.
- `NUMERICAL`: one numerical value per object.
- `NUMERICAL_LIST`: several numerical values per object, such as a text embedding.


A movie can have several genres, so `genre` is a `CATEGORICAL_LIST` feature. The source file stores genres in one string. Split that string into a list first.


In [9]:
item_features_encoded["genre"] = item_features_encoded["genre"].str.split("|", regex=False)
item_features_encoded

,movie_title,genre,item_id
0,Toy Story (1995),"[Animation, Children's, Comedy]",0
1,Jumanji (1995),"[Adventure, Children's, Fantasy]",1
2,Grumpier Old Men (1995),"[Comedy, Romance]",2
3,Waiting to Exhale (1995),"[Comedy, Drama]",3
4,Father of the Bride Part II (1995),[Comedy],4
...,...,...,...
3878,Meet the Parents (2000),[Comedy],3878
3879,Requiem for a Dream (2000),[Drama],3879
3880,Tigerland (2000),[Drama],3880
3881,Two Family House (2000),[Drama],3881


Encode the genre lists with `SequenceEncodingRule`.


In [10]:
from replay.preprocessing import SequenceEncodingRule

genre_encoder = LabelEncoder([SequenceEncodingRule("genre")])
item_features_encoded = genre_encoder.fit_transform(item_features_encoded)
item_features_encoded.head(5)

,movie_title,item_id,genre
0,Toy Story (1995),0,"[2, 3, 4]"
1,Jumanji (1995),1,"[1, 3, 8]"
2,Grumpier Old Men (1995),2,"[4, 13]"
3,Waiting to Exhale (1995),3,"[4, 7]"
4,Father of the Bride Part II (1995),4,[4]


Save the longest genre list and the number of unique genres. These values are needed later in the metadata for `ParquetModule` and `ItemTower`.


In [11]:
MAX_LEN_GENRE = int(item_features_encoded["genre"].apply(len).max())
NUM_UNIQUE_GENRE_VALUES = int(len(genre_encoder.mapping["genre"]))

`movie_title` is a text feature. There are two common ways to encode it:

- Use `CountVectorizer` or a similar method. The result is a `CATEGORICAL_LIST` feature with a categorical embedding.
- Use a pretrained text model. The result is a `NUMERICAL_LIST` feature. If its size differs from the model embedding size, RePlay adds a linear projection. If the sizes match, the values are used directly.

This example uses the second option with a model from `sentence-transformers`.


In [12]:
%pip install sentence-transformers -q


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer

text_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = item_features_encoded["movie_title"].fillna("").tolist()
embeddings = text_encoder.encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
item_features_encoded["movie_title"] = [value.astype(np.float32) for value in embeddings]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
item_features_encoded.head(5)

,movie_title,item_id,genre
0,"[-0.082835145, 0.053030428, 0.053576548, -0.02...",0,"[2, 3, 4]"
1,"[-0.10529309, 0.15084095, -0.02639786, -0.0655...",1,"[1, 3, 8]"
2,"[-0.09878677, 0.017649995, -0.05274402, -0.038...",2,"[4, 13]"
3,"[-0.08723061, 0.03661162, -0.021703225, -0.012...",3,"[4, 7]"
4,"[-0.0692064, 0.03875223, 0.01473796, 0.0121407...",4,[4]


In [15]:
TITLE_EMB_DIM = int(text_encoder.get_embedding_dimension())

### Build sequential datasets

Join the encoded item features to the interactions. Then group each user's events into a sequence sorted by time.


In [16]:
train_events = train_events.merge(item_features_encoded, on="item_id", how="inner")
val_events = val_events.merge(item_features_encoded, on="item_id", how="inner")
test_events = test_events.merge(item_features_encoded, on="item_id", how="inner")

In [17]:
from replay.data.nn.utils import groupby_sequences


def bake_data(df: pd.DataFrame):
    return groupby_sequences(events=df, groupby_col="user_id", sort_col="timestamp")


train_events = bake_data(train_events)
val_events = bake_data(val_events)
val_gt = bake_data(val_gt)
test_events = bake_data(test_events)
test_gt = bake_data(test_gt)

train_events.head(3)


,user_id,timestamp,item_id,movie_title,genre
0,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[3117, 1009, 1250, 1672, 2271, 1768, 3339, 118...","[[-0.11641536, -0.063095465, 0.0054702833, -0....","[[7], [2, 3, 11], [4, 14], [7, 13], [13], [7],..."
1,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1180, 1199, 1192, 2648, 1273, 2874, 1207, 117...","[[-0.027989453, 0.07571321, -0.06741784, 0.025...","[[0, 1], [7, 16], [0, 1, 13, 14, 16], [4, 10],..."
2,2,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[589, 2789, 3465, 1899, 1892, 1407, 1246, 1358...","[[-0.07848771, -0.00096751953, -0.019314129, 0...","[[7, 15], [4, 7], [4], [4, 7], [7], [0, 4], [1..."


Add `ground_truth` to the validation events. `ParquetModule` uses it to calculate validation metrics.


In [18]:
def add_gt_to_events(events_df, gt_df):
    gt_to_join = gt_df[["user_id", "item_id"]].rename(columns={"item_id": "ground_truth"})

    events_df = events_df.merge(gt_to_join, on="user_id", how="inner")
    return events_df


val_events = add_gt_to_events(val_events, val_gt)
val_events.head(3)

,user_id,timestamp,item_id,movie_title,genre,ground_truth
0,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[3117, 1009, 1250, 1672, 2271, 1768, 3339, 118...","[[-0.11641536, -0.063095465, 0.0054702833, -0....","[[7], [2, 3, 11], [4, 14], [7, 13], [13], [7],...",[1838]
1,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1180, 1199, 1192, 2648, 1273, 2874, 1207, 117...","[[-0.027989453, 0.07571321, -0.06741784, 0.025...","[[0, 1], [7, 16], [0, 1, 13, 14, 16], [4, 10],...",[1848]
2,2,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[589, 2789, 3465, 1899, 1892, 1407, 1246, 1358...","[[-0.07848771, -0.00096751953, -0.019314129, 0...","[[7, 15], [4, 7], [4], [4, 7], [7], [0, 4], [1...",[102]


The two towers read their data separately:

- The query tower reads `TRAIN_PATH`, `VAL_PATH`, and `PREDICT_PATH` one batch at a time.
- The item tower loads all encoded item features from `PATH_ENCODED_FEATURES` and keeps them in memory during training and inference.

`ItemTower` expects features that are already encoded.


In [19]:
from pathlib import Path

data_dir = Path("temp/preprocessed/")
data_dir.mkdir(parents=True, exist_ok=True)

PATH_ENCODED_FEATURES = data_dir / "item_features_encoded.parquet"

TRAIN_PATH = data_dir / "train.parquet"
VAL_PATH = data_dir / "val.parquet"
PREDICT_PATH = data_dir / "test.parquet"

ENCODER_PATH = data_dir / "encoder"
GENRE_ENCODER_PATH = data_dir / "genre_encoder"

In [20]:
item_features_names =  ["item_id", "genre", "movie_title"]
item_features_encoded[item_features_names].to_parquet(PATH_ENCODED_FEATURES)

events_features_names = item_features_names + ["user_id"]

train_events[events_features_names].to_parquet(TRAIN_PATH)
val_events[events_features_names + ["ground_truth"]].to_parquet(VAL_PATH)
test_events[events_features_names].to_parquet(PREDICT_PATH)

encoder.save(ENCODER_PATH)
genre_encoder.save(GENRE_ENCODER_PATH)

/home/evtsinovnik/RePlay/replay/preprocessing/label_encoder.py:1002: UserWarning: There is already LabelEncoder object saved at the given path. File will be overwrited.
  warnings.warn(msg)
/home/evtsinovnik/RePlay/replay/preprocessing/label_encoder.py:563: UserWarning: There is already LabelEncodingRule object saved at the given path. File will be overwrited.
  warnings.warn(msg)


## Prepare the training setup

### Define the tensor schema

`TensorSchema` contains only the features used by the model. Categorical features need `cardinality` and `padding_value`. Numerical lists need `tensor_dim`. The `feature_sources` field tells `TwoTower` where each item feature comes from.


In [21]:
from replay.data import FeatureHint, FeatureSource, FeatureType
from replay.data.nn import TensorFeatureInfo, TensorFeatureSource, TensorSchema

EMBEDDING_DIM = 192
NUM_UNIQUE_ITEMS = len(encoder.mapping["item_id"])

tensor_schema = TensorSchema(
    [
        TensorFeatureInfo(
            "item_id",
            is_seq=True,
            feature_type=FeatureType.CATEGORICAL,
            embedding_dim=EMBEDDING_DIM,
            padding_value=NUM_UNIQUE_ITEMS,
            cardinality=NUM_UNIQUE_ITEMS,
            feature_hint=FeatureHint.ITEM_ID,
            feature_sources=[TensorFeatureSource(FeatureSource.INTERACTIONS, "item_id")],
        ),
        TensorFeatureInfo(
            "genre",
            is_seq=True,
            feature_type=FeatureType.CATEGORICAL_LIST,
            embedding_dim=EMBEDDING_DIM,
            padding_value=NUM_UNIQUE_GENRE_VALUES,
            cardinality=NUM_UNIQUE_GENRE_VALUES,
            feature_sources=[TensorFeatureSource(FeatureSource.ITEM_FEATURES, "genre")],
        ),
        TensorFeatureInfo(
            "movie_title",
            is_seq=True,
            feature_type=FeatureType.NUMERICAL_LIST,
            embedding_dim=EMBEDDING_DIM,
            tensor_dim=TITLE_EMB_DIM,
            feature_sources=[TensorFeatureSource(FeatureSource.ITEM_FEATURES, "movie_title")],
        ),
    ]
)

### Configure `ParquetModule` transforms

The training pipeline uses additional two pairs of components for optimization:

1. `GroupedUniformNegativeSamplingTransform` creates an independent negative pool for each logical group. Its `group_size` must match `logical_batch_size` in `GroupedCESampled`.
2. `TargetAwareAdaptiveTrimTransform` removes left padding that is not used by inputs or targets. `LengthBucketedQueryEncoder` then reduces the sequence length again inside groups of similarly sized sequences.

The transforms are listed explicitly so that their order is easy to see.

In [22]:
from replay.nn.transform import (
    GroupedUniformNegativeSamplingTransform,
    GroupTransform,
    NextTokenTransform,
    RenameTransform,
    TargetAwareAdaptiveTrimTransform,
    UnsqueezeTransform,
)

LOGICAL_BATCH_SIZE = 32
NUM_NEGATIVE_SAMPLES = min(1_000, NUM_UNIQUE_ITEMS - 1)
ITEM_ID_NAME = tensor_schema.item_id_feature_name

train_transforms = [
    NextTokenTransform(label_name=ITEM_ID_NAME, shift=1),
    RenameTransform(
        {
            f"{ITEM_ID_NAME}_mask": "padding_mask",
            "positive_labels_mask": "target_padding_mask",
        }
    ),
    # Both masks must still be two-dimensional at this point.
    TargetAwareAdaptiveTrimTransform(
        feature_names=[*tensor_schema.sequential_features.names, "positive_labels"],
    ),
    UnsqueezeTransform("target_padding_mask", -1),
    UnsqueezeTransform("positive_labels", -1),
    GroupTransform({"feature_tensors": tensor_schema.names}),
    GroupedUniformNegativeSamplingTransform(
        cardinality=NUM_UNIQUE_ITEMS,
        num_negative_samples=NUM_NEGATIVE_SAMPLES,
        group_size=LOGICAL_BATCH_SIZE,
    ),
]


def make_inference_transforms():
    return [
        RenameTransform({f"{ITEM_ID_NAME}_mask": "padding_mask"}),
        GroupTransform({"feature_tensors": tensor_schema.names}),
    ]


transforms = {
    "train": train_transforms,
    "validate": make_inference_transforms(),
    "test": make_inference_transforms(),
    "predict": make_inference_transforms(),
}


Metadata describes every input column for each data split. `shape` tells `ParquetModule` how large a tensor must be. Longer values are trimmed, and shorter values are padded on the left with `padding`.

Use plain Python `int` values in metadata, not values such as `numpy.int64`.

For `CATEGORICAL` and `NUMERICAL` sequence features, `shape` is the sequence length. For `CATEGORICAL_LIST` and `NUMERICAL_LIST` sequence features, use `[sequence_length, feature_size]`.


In [23]:
def create_meta(shape: int, gt_shape: int | None = None):
    meta = {
        "item_id": {"shape": shape, "padding": tensor_schema["item_id"].padding_value},
        "genre": {
            "shape": [shape, MAX_LEN_GENRE],
            "padding": tensor_schema["genre"].padding_value,
        },
        "movie_title": {
            "shape": [shape, TITLE_EMB_DIM],
            "padding": tensor_schema["movie_title"].padding_value,
        },
    }
    if gt_shape is not None:
        meta.update({"ground_truth": {"shape": gt_shape, "padding": -1}})
    return meta

For the training split, set the sequence length to `MAX_SEQ_LEN + shift`. `NextTokenTransform` needs the extra item to create the shifted next-item targets. This example uses `shift=1`.


In [24]:
val_gt_len = int(val_events["ground_truth"].apply(len).max())

MAX_SEQ_LEN = 50

TRAIN_METADATA = {
    "train": create_meta(shape=MAX_SEQ_LEN + 1),
    "validate": create_meta(shape=MAX_SEQ_LEN, gt_shape=val_gt_len),
}
INFERENCE_METADATA = {"predict": create_meta(shape=MAX_SEQ_LEN) | {"user_id": {}}}

In [25]:
from replay.data.nn import ParquetModule

# Several logical groups are packed into one physical data-loader batch.
BATCH_SIZE = 4 * LOGICAL_BATCH_SIZE

train_parquet_datamodule = ParquetModule(
    train_path=TRAIN_PATH,
    validate_path=VAL_PATH,
    batch_size=BATCH_SIZE,
    metadata=TRAIN_METADATA,
    transforms=transforms,
)

/tmp/ipykernel_8198/1358046222.py:6: UserWarning: The following dataset paths aren't provided: test,predict. Make sure to disable these stages in your Lightning Trainer configuration.
  train_parquet_datamodule = ParquetModule(


You can inspect one raw training batch while debugging.


In [26]:
train_parquet_datamodule.setup("train")
batch = next(iter(train_parquet_datamodule.train_dataloader()))

In [27]:
def print_tensor_shapes(d: dict, indent: int = 0):
    for key, value in d.items():
        if isinstance(value, dict):
            print(" " * indent + f"{key}:")
            print_tensor_shapes(value, indent + 2)
        elif isinstance(value, torch.Tensor):
            print(" " * indent + f"{key}: shape={tuple(value.shape)}")
        else:
            print(" " * indent + f"{key}: (not a tensor)")

In [28]:
print_tensor_shapes(batch)

item_id_mask: shape=(128, 51)
item_id: shape=(128, 51)
genre_mask: shape=(128, 51, 6)
genre: shape=(128, 51, 6)
movie_title_mask: shape=(128, 51, 384)
movie_title: shape=(128, 51, 384)


You can also inspect the same batch after the training transforms are applied.


In [29]:
preprocessed_batch = train_parquet_datamodule.compiled_transforms["train"](batch)

In [30]:
print_tensor_shapes(preprocessed_batch)

padding_mask: shape=(128, 50)
genre_mask: shape=(128, 50, 6)
movie_title_mask: shape=(128, 50, 384)
target_padding_mask: shape=(128, 50, 1)
positive_labels: shape=(128, 50, 1)
feature_tensors:
  item_id: shape=(128, 50)
  genre: shape=(128, 50, 6)
  movie_title: shape=(128, 50, 384)
negative_labels: shape=(4, 1000)


## Train the model

### Build the TwoTower model

The model instance is builded from individual blocks because this training setup needs a custom query encoder and a custom loss which differ from standard ones in `from_params` method. The same `build_twotower` function is used for training and checkpoint loading, so both stages always use the same architecture.

`FeaturesReader` loads the complete item catalog. Its metadata follows the same basic rules as `ParquetModule` metadata, but catalog rows are not sequences. Use an empty dictionary for scalar features and specify `shape` only for list features.

During validation or inference, `TwoTower` builds an item cache so that item embeddings are computed only once. The `item_cache_batch_size` parameter is optional. If it is not set, the entire catalog is encoded in one pass and the full item cache is kept in memory. Set this parameter to build the cache in smaller chunks when encoding the entire catalog at once would exceed GPU memory. Chunking reduces peak memory use during cache construction, but makes this one-time operation slower. It does not change the size of the completed cache. This example sets `ITEM_CACHE_BATCH_SIZE` below and passes it to `TwoTowerBody`.


In [31]:
feature_reader_metadata = {
    "item_id": {},
    "genre": {"shape": MAX_LEN_GENRE, "padding": tensor_schema["genre"].padding_value},
    "movie_title": {
        "shape": TITLE_EMB_DIM,
        "padding": tensor_schema["movie_title"].padding_value,
    },
}

In [32]:
from replay.nn.agg import SumAggregator
from replay.nn.embedding import SequenceEmbedding
from replay.nn.ffn import SwiGLUEncoder
from replay.nn.loss import GroupedCESampled
from replay.nn.mask import DefaultAttentionMask
from replay.nn.sequential import LengthBucketedQueryEncoder, PositionAwareAggregator, SasRecTransformerLayer, TwoTower
from replay.nn.sequential.twotower import FeaturesReader, TwoTowerBody

NUM_HEADS = 2
NUM_BLOCKS = 2
DROPOUT = 0.2
LENGTH_BUCKET_SIZE = 32
ITEM_CACHE_BATCH_SIZE = 512


def build_twotower() -> TwoTower:
    common_aggregator = SumAggregator(embedding_dim=EMBEDDING_DIM)
    query_encoder = LengthBucketedQueryEncoder(
        encoder=SasRecTransformerLayer(
            embedding_dim=EMBEDDING_DIM,
            num_heads=NUM_HEADS,
            num_blocks=NUM_BLOCKS,
            dropout=DROPOUT,
            activation="relu",
        ),
        bucket_size=LENGTH_BUCKET_SIZE,
        sequence_shift=1,
    )
    body = TwoTowerBody(
        schema=tensor_schema,
        embedder=SequenceEmbedding(schema=tensor_schema),
        attn_mask_builder=DefaultAttentionMask(
            reference_feature_name=tensor_schema.item_id_feature_name,
            num_heads=NUM_HEADS,
        ),
        query_tower_feature_names=tensor_schema.names,
        query_embedding_aggregator=PositionAwareAggregator(
            embedding_aggregator=common_aggregator,
            max_sequence_length=MAX_SEQ_LEN,
            dropout=DROPOUT,
        ),
        item_embedding_aggregator=common_aggregator,
        query_encoder=query_encoder,
        query_tower_output_normalization=torch.nn.LayerNorm(EMBEDDING_DIM),
        item_encoder=SwiGLUEncoder(
            embedding_dim=EMBEDDING_DIM,
            hidden_dim=2 * EMBEDDING_DIM,
        ),
        item_features_reader=FeaturesReader(
            schema=tensor_schema,
            metadata=feature_reader_metadata,
            path=PATH_ENCODED_FEATURES,
        ),
        item_cache_batch_size=ITEM_CACHE_BATCH_SIZE,
    )
    loss = GroupedCESampled(
        logical_batch_size=LOGICAL_BATCH_SIZE,
        cardinality=NUM_UNIQUE_ITEMS,
    )
    return TwoTower(body=body, loss=loss)


twotower = build_twotower()

`LightningModule` provides the training, validation, and prediction loops for the `TwoTower` model.


In [33]:
from replay.nn.lightning import LightningModule

model = LightningModule(twotower)

`ComputeMetricsCallback` calculates validation metrics. Pass `train_parquet_datamodule` to `Trainer.fit` to use the prepared datasets and transforms.


In [34]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from replay.nn.lightning.callback import ComputeMetricsCallback

checkpoint_callback = ModelCheckpoint(
    dirpath="/home/RePlay/examples/twotower/.checkpoints",
    save_top_k=1,
    verbose=True,
    monitor="recall@10",
    mode="max",
)

validation_metrics_callback = ComputeMetricsCallback(
    metrics=["map", "ndcg", "recall"],
    ks=[1, 5, 10, 20],
    item_count=NUM_UNIQUE_ITEMS,
)

csv_logger = CSVLogger(save_dir="/home/RePlay/examples/twotower/.logs/train")

trainer = L.Trainer(
    max_epochs=5,
    callbacks=[checkpoint_callback, validation_metrics_callback],
    logger=csv_logger,
    enable_progress_bar=False,
)

trainer.fit(model, datamodule=train_parquet_datamodule)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ TwoTower │  1.7 M │ train │     0 │
└───┴───────┴──────────┴────────┴───────┴───────┘

Trainable params: 1.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.7 M                                                                                                
Total estimated model params size (MB): 6.897                                                                      
Modules in train mode: 58                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/replay/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/replay/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.
/home/replay/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py:321: The number of training batches (48) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
/home/replay/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 128. To avoid any miscalculations, use `se

k             1         5         10        20
map     0.003311  0.007376  0.009121  0.011308
ndcg    0.003311  0.009499  0.013760  0.021938
recall  0.003311  0.016060  0.029305  0.062086 



...

k             1         5         10        20
map     0.005132  0.012216  0.016054  0.019392
ndcg    0.005132  0.015988  0.025361  0.037753
recall  0.005132  0.027649  0.056788  0.106291 



Epoch 3, global step 192: 'recall@10' reached 0.07152 (best 0.07152), saving model to '/home/RePlay/examples/twotower/.checkpoints/epoch=3-step=192.ckpt' as top 1


k            1         5         10        20
map     0.00745  0.017158  0.021300  0.025109
ndcg    0.00745  0.022501  0.032807  0.047219
recall  0.00745  0.039073  0.071523  0.129636 



Epoch 4, global step 240: 'recall@10' reached 0.08427 (best 0.08427), saving model to '/home/RePlay/examples/twotower/.checkpoints/epoch=4-step=240.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=5` reached.


k             1         5         10        20
map     0.006291  0.017139  0.022741  0.027423
ndcg    0.006291  0.022873  0.036773  0.054173
recall  0.006291  0.040563  0.084272  0.153808 



In [35]:
best_model_path = checkpoint_callback.best_model_path
best_model_path

'/home/RePlay/examples/twotower/.checkpoints/epoch=4-step=240.ckpt'

## Run inference and calculate metrics

Load the best checkpoint with `LightningModule.load_from_checkpoint`.

The `model` argument must have the same architecture as the model used for training. The method loads the saved weights into this model.


In [36]:
twotower = build_twotower()

best_model = LightningModule.load_from_checkpoint(best_model_path, model=twotower, weights_only=False)
best_model.eval()

LightningModule(
  (model): TwoTower(
    (body): TwoTowerBody(
      (embedder): SequenceEmbedding(
        (feature_embedders): ModuleDict(
          (item_id): CategoricalEmbedding(
            (emb): Embedding(3884, 192, padding_idx=3883)
          )
          (genre): CategoricalEmbedding(
            (emb): EmbeddingBag(19, 192, mode='sum', padding_idx=18)
          )
          (movie_title): NumericalEmbedding(
            (linear): Linear(in_features=384, out_features=192, bias=True)
          )
        )
      )
      (query_tower): QueryTower(
        (embedder): SequenceEmbedding(
          (feature_embedders): ModuleDict(
            (item_id): CategoricalEmbedding(
              (emb): Embedding(3884, 192, padding_idx=3883)
            )
            (genre): CategoricalEmbedding(
              (emb): EmbeddingBag(19, 192, mode='sum', padding_idx=18)
            )
            (movie_title): NumericalEmbedding(
              (linear): Linear(in_features=384, out_features=192

In [37]:
inference_parquet_module = ParquetModule(
    predict_path=PREDICT_PATH,
    batch_size=BATCH_SIZE,
    metadata=INFERENCE_METADATA,
    transforms=transforms,
)

/tmp/ipykernel_8198/2294316212.py:1: UserWarning: The following dataset paths aren't provided: train,validate,test. Make sure to disable these stages in your Lightning Trainer configuration.
  inference_parquet_module = ParquetModule(


`PandasTopItemsCallback` collects the highest-scoring items in a Pandas DataFrame.

RePlay also provides `PolarsTopItemsCallback`, `SparkTopItemsCallback`, and `TorchTopItemsCallback` for other output formats.


In [38]:
from replay.nn.lightning.callback import PandasTopItemsCallback

csv_logger = CSVLogger(save_dir=".logs/test", name="TwoTower-example")

TOPK = [1, 5, 10, 20]

pandas_prediction_callback = PandasTopItemsCallback(
    top_k=max(TOPK),
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
)

trainer = L.Trainer(
    callbacks=[pandas_prediction_callback],
    logger=csv_logger,
    inference_mode=True,
    enable_progress_bar=False,
)

trainer.predict(best_model, datamodule=inference_parquet_module, return_predictions=False)

pandas_res = pandas_prediction_callback.get_result()

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


/home/replay/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/replay/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.


### Calculate metrics


In [39]:
from replay.metrics import MAP, OfflineMetrics, Precision, Recall
from replay.metrics.torch_metrics_builder import metrics_to_df

result_metrics = OfflineMetrics(
    [Recall(TOPK), Precision(TOPK), MAP(TOPK)],
    query_column="user_id",
    rating_column="score",
)(pandas_res, test_gt[["user_id", "item_id"]].explode("item_id"))

metrics_to_df(result_metrics)

/home/evtsinovnik/RePlay/replay/metrics/offline_metrics.py:374: UserWarning: ground_truth contains queries that are not presented in recommendations
  warnings.warn(f"{dataset_name} contains queries that are not presented in recommendations")


k,1,5,10,20
MAP,0.007287,0.020388,0.025713,0.030513
Precision,0.007287,0.009738,0.009043,0.008032
Recall,0.007287,0.048692,0.090427,0.160649


Use the fitted `LabelEncoder` to convert recommendation IDs back to their original values.


In [40]:
encoder = LabelEncoder.load(ENCODER_PATH)
encoder.inverse_transform(pandas_res)

,user_id,item_id,score
0,1,364,88.589455
0,1,588,88.331085
0,1,595,88.304787
0,1,2355,88.184319
0,1,1907,88.011856
...,...,...,...
6039,6040,1036,98.461975
6039,6040,3147,98.430801
6039,6040,1208,98.424789
6039,6040,1196,98.395317
